# Transformer dalam AI

Transformer adalah arsitektur jaringan saraf yang memodelkan hubungan antar-token melalui attention, bukan melalui pemrosesan sekuensial seperti RNN. Pada repositori ini, transformer dipaketkan ke dalam wrapper yang memudahkan penggunaan untuk model decoder-only, encoder-only, dan encoder-decoder.

Secara konseptual, alurnya adalah sebagai berikut:
1. Token diubah menjadi embedding.
2. Posisi token diinformasikan ke model.
3. Self-attention menghitung keterkaitan antartoken.
4. Feedforward network memperkaya representasi.
5. Residual connection dan normalisasi menjaga stabilitas pelatihan.

Dalam praktik di repo ini, kelas seperti `TransformerWrapper`, `Encoder`, `Decoder`, dan `XTransformer` menjadi titik masuk utama untuk membangun model yang berbeda sesuai kebutuhan.

## Komponen Utama

| Komponen | Fungsi | Relevansi di repositori |
| --- | --- | --- |
| Embedding | Mengubah token menjadi vektor padat | Dibangun di dalam `TransformerWrapper` |
| Self-attention | Mengukur hubungan antartoken dalam satu urutan | Inti dari `Encoder` dan `Decoder` |
| Cross-attention | Menghubungkan decoder dengan konteks dari encoder | Aktif saat `cross_attend=True` |
| Feedforward | Memberi transformasi non-linear pada representasi | Melengkapi setiap blok transformer |
| Residual dan normalisasi | Menjaga aliran gradien dan stabilitas | Dipakai dalam blok utama model |
| Masking | Membatasi token mana yang boleh saling melihat | Penting untuk padding dan autoregresi |

```mermaid
flowchart LR
    A[Token masuk] --> B[Embedding]
    B --> C[Informasi posisi]
    C --> D[Self-attention multi-head]
    D --> E[Residual + normalisasi]
    E --> F[Feedforward]
    F --> G[Logits]
```

Diagram di atas menunjukkan bahwa transformer tidak bekerja dengan urutan komputasi yang ketat per langkah waktu, melainkan memproses relasi token secara paralel. Ini yang membuatnya efisien untuk menangkap dependensi jarak jauh.

In [ ]:
import torch
from x_transformers import TransformerWrapper, Decoder

torch.manual_seed(7)

model = TransformerWrapper(
    num_tokens = 64,
    max_seq_len = 16,
    attn_layers = Decoder(
        dim = 128,
        depth = 2,
        heads = 4,
    )
)

tokens = torch.randint(0, 64, (2, 16))
mask = torch.ones_like(tokens).bool()
mask[:, -4:] = False

logits = model(tokens, mask = mask)

print("Input shape:", tokens.shape)
print("Mask shape:", mask.shape)
print("Output shape:", logits.shape)
print("Parameter count:", sum(p.numel() for p in model.parameters()))

In [ ]:
from x_transformers import XTransformer

seq2seq = XTransformer(
    dim = 128,
    enc_num_tokens = 64,
    enc_depth = 2,
    enc_heads = 4,
    enc_max_seq_len = 16,
    dec_num_tokens = 64,
    dec_depth = 2,
    dec_heads = 4,
    dec_max_seq_len = 16,
)

src = torch.randint(0, 64, (2, 16))
tgt = torch.randint(0, 64, (2, 16))
src_mask = torch.ones_like(src).bool()
loss = seq2seq(src, tgt, mask = src_mask)

print("Seq2seq loss:", float(loss))

## Ringkasan Praktis

- `TransformerWrapper` cocok untuk pemodelan token tunggal dengan skema decoder-only atau encoder-only.
- `XTransformer` cocok untuk tugas berpasangan, misalnya terjemahan atau ringkasan.
- `Decoder` dipakai saat model harus memprediksi token berikutnya secara autoregresif.
- `Encoder` dipakai saat keluaran yang dibutuhkan adalah representasi konteks, bukan langsung generasi token.

Dengan demikian, transformer dalam repositori ini bukan sekadar satu model, melainkan keluarga komponen yang dapat dirakit sesuai jenis tugas.

## Implementasi Proyek

Proyek ini membangun pengklasifikasi sentimen teks sederhana dengan transformer. Dataset dibuat langsung di notebook agar contoh tetap mandiri, mudah dibaca, dan tidak bergantung pada berkas eksternal.

Alur proyek:
1. Menyusun korpus kalimat positif dan negatif.
2. Membangun kamus token.
3. Melatih `TransformerWrapper` dengan `Encoder` dan token klasifikasi internal.
4. Menguji prediksi pada kalimat baru.

In [ ]:
import random
from collections import Counter

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from x_transformers import TransformerWrapper, Encoder

seed = 7
random.seed(seed)
torch.manual_seed(seed)

positive_texts = [
    "film ini sangat bagus",
    "alur ceritanya menarik",
    "akting para pemain luar biasa",
    "saya suka dialognya",
    "musik latarnya indah",
    "pengalaman menonton yang menyenangkan",
    "ceritanya hangat dan kuat",
    "penyutradaraannya rapi",
]

negative_texts = [
    "film ini sangat membosankan",
    "alur ceritanya lemah",
    "aktingnya terasa kaku",
    "saya tidak suka dialognya",
    "musik latarnya mengganggu",
    "pengalaman menonton yang buruk",
    "ceritanya panjang dan datar",
    "penyutradaraannya berantakan",
]

samples = [(text, 1) for text in positive_texts] + [(text, 0) for text in negative_texts]
random.shuffle(samples)

split_index = int(len(samples) * 0.75)
train_samples = samples[:split_index]
valid_samples = samples[split_index:]

max_seq_len = 8
special_tokens = ["<pad>", "<unk>"]

counter = Counter()
for text, _ in samples:
    counter.update(text.lower().split())

vocab = {token: index for index, token in enumerate(special_tokens + sorted(counter))}
pad_id = vocab["<pad>"]
unk_id = vocab["<unk>"]


def tokenize(text):
    return text.lower().split()


def encode(text):
    return [vocab.get(token, unk_id) for token in tokenize(text)]


class TextDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]


def collate_batch(batch):
    texts, labels = zip(*batch)
    encoded_texts = [torch.tensor(encode(text)[:max_seq_len], dtype=torch.long) for text in texts]
    batch_size = len(encoded_texts)
    current_seq_len = max(seq.numel() for seq in encoded_texts)

    tokens = torch.full((batch_size, current_seq_len), pad_id, dtype=torch.long)
    mask = torch.zeros((batch_size, current_seq_len), dtype=torch.bool)

    for row_index, sequence in enumerate(encoded_texts):
        length = sequence.numel()
        tokens[row_index, :length] = sequence
        mask[row_index, :length] = True

    labels = torch.tensor(labels, dtype=torch.long)
    return tokens, mask, labels


train_loader = DataLoader(TextDataset(train_samples), batch_size=4, shuffle=True, collate_fn=collate_batch)
valid_loader = DataLoader(TextDataset(valid_samples), batch_size=4, shuffle=False, collate_fn=collate_batch)

print("Jumlah kosakata:", len(vocab))
print("Data latih:", len(train_samples))
print("Data validasi:", len(valid_samples))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerWrapper(
    num_tokens = len(vocab),
    max_seq_len = max_seq_len,
    logits_dim = 2,
    use_cls_token = True,
    attn_layers = Encoder(
        dim = 64,
        depth = 2,
        heads = 4,
    )
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr = 3e-3, weight_decay = 1e-2)


def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_items = 0

    for tokens, mask, labels in loader:
        tokens = tokens.to(device)
        mask = mask.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        logits = model(tokens, mask = mask)
        loss = criterion(logits, labels)

        if training:
            loss.backward()
            optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim = -1) == labels).sum().item()
        total_items += batch_size

    return total_loss / total_items, total_correct / total_items


epochs = 25
history = []

for epoch in range(1, epochs + 1):
    train_loss, train_acc = run_epoch(train_loader, True)
    valid_loss, valid_acc = run_epoch(valid_loader, False)
    history.append((train_loss, train_acc, valid_loss, valid_acc))

    if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
        print(
            f"Epoch {epoch:02d} | "
            f"train loss {train_loss:.4f} | train acc {train_acc:.2%} | "
            f"valid loss {valid_loss:.4f} | valid acc {valid_acc:.2%}"
        )

In [ ]:
def predict(text):
    model.eval()
    encoded = encode(text)[:max_seq_len]

    tokens = torch.full((1, max_seq_len), pad_id, dtype=torch.long, device = device)
    mask = torch.zeros((1, max_seq_len), dtype=torch.bool, device = device)

    if encoded:
        sequence = torch.tensor(encoded, dtype=torch.long, device = device)
        length = sequence.numel()
        tokens[0, :length] = sequence
        mask[0, :length] = True

    with torch.no_grad():
        logits = model(tokens, mask = mask)
        probabilities = logits.softmax(dim = -1)[0]
        label = int(probabilities.argmax().item())

    return {
        "text": text,
        "label": "positif" if label == 1 else "negatif",
        "probability": float(probabilities[label])
    }


examples = [
    "film ini sangat bagus",
    "ceritanya panjang dan datar",
    "saya suka dialognya",
]

for text in examples:
    result = predict(text)
    print(f"{result['text']} -> {result['label']} ({result['probability']:.3f})")